# Architecture Tour: GPT → LLaMA → Hybrid

Walk through three architectures side-by-side. For each:
config → model → forward pass → parameter counting.

**Prerequisites:** `uv sync --extra dev`

In [ ]:
import mlx.core as mx
import mlx.utils

from lmxlab.models.base import LanguageModel
from lmxlab.models.gpt import gpt_tiny
from lmxlab.models.llama import llama_tiny
from lmxlab.models.falcon import falcon_h1_tiny

## 1. GPT: The Classic Transformer

GPT ([Radford et al., 2019](https://cdn.openai.com/better-language-models/language_models_are_unsupervised_multitask_learners.pdf))
uses: LayerNorm, standard multi-head attention
([Vaswani et al., 2017](https://arxiv.org/abs/1706.03762)),
standard FFN (GELU), sinusoidal position encoding, pre-norm,
bias everywhere.

In [ ]:
gpt_cfg = gpt_tiny()
print(f"Block config:")
print(f"  Attention: {gpt_cfg.block.attention}")
print(f"  FFN:       {gpt_cfg.block.ffn}")
print(f"  Norm:      {gpt_cfg.block.norm}")
print(f"  Position:  {gpt_cfg.block.position}")
print(f"  d_model:   {gpt_cfg.block.d_model}")
print(f"  n_heads:   {gpt_cfg.block.n_heads}")
print(f"  d_ff:      {gpt_cfg.block.d_ff}")
print(f"  Bias:      {gpt_cfg.block.bias}")
print(f"\nModel config:")
print(f"  n_layers:  {gpt_cfg.n_layers}")
print(f"  vocab:     {gpt_cfg.vocab_size}")

In [ ]:
gpt = LanguageModel(gpt_cfg)
mx.eval(gpt.parameters())
print(f"GPT params: {gpt.count_parameters():,}")

# Forward pass
tokens = mx.zeros((1, 16), dtype=mx.int32)
logits, _ = gpt(tokens)
mx.eval(logits)
print(f"Input:  {tokens.shape}  (batch, seq_len)")
print(f"Output: {logits.shape}  (batch, seq_len, vocab)")

## 2. LLaMA: Modern Best Practices

LLaMA ([Touvron et al., 2023](https://arxiv.org/abs/2302.13971))
uses: RMSNorm ([Zhang & Sennrich, 2019](https://arxiv.org/abs/1910.07467)),
GQA ([Ainslie et al., 2023](https://arxiv.org/abs/2305.13245))
with fewer KV heads, gated FFN with SwiGLU activation
([Shazeer, 2020](https://arxiv.org/abs/2002.05202)),
RoPE position encoding ([Su et al., 2021](https://arxiv.org/abs/2104.09864)),
no bias.

In [ ]:
llama_cfg = llama_tiny()
print(f"LLaMA differences from GPT:")
print(f"  Attention: {llama_cfg.block.attention} (GQA with {llama_cfg.block.n_kv_heads} KV heads)")
print(f"  FFN:       {llama_cfg.block.ffn} (SwiGLU — 3 projections)")
print(f"  Norm:      {llama_cfg.block.norm} (no mean subtraction)")
print(f"  Position:  {llama_cfg.block.position} (rotary embeddings)")
print(f"  Bias:      {llama_cfg.block.bias} (removed entirely)")

In [ ]:
llama = LanguageModel(llama_cfg)
mx.eval(llama.parameters())
print(f"LLaMA params: {llama.count_parameters():,}")

logits, _ = llama(tokens)
mx.eval(logits)
print(f"Output: {logits.shape}")

## 3. Falcon-H1: SSM/Attention Hybrid

Falcon-H1 ([Zuo et al., 2025](https://arxiv.org/abs/2507.22448))
interleaves Mamba-2 SSM layers
([Dao & Gu, 2024](https://arxiv.org/abs/2405.21060)) and GQA
layers in a MMM\* pattern: 3 Mamba blocks for every 1 attention
block. SSM layers have O(n) complexity vs O(n^2) for attention.

In [ ]:
falcon_cfg = falcon_h1_tiny()
print(f"Falcon-H1 layer types:")
for i in range(falcon_cfg.n_layers):
    bcfg = falcon_cfg.get_block_config(i)
    layer_type = "SSM (Mamba-2)" if "mamba" in bcfg.attention else "Attention (GQA)"
    print(f"  Layer {i}: {layer_type}")

In [ ]:
falcon = LanguageModel(falcon_cfg)
mx.eval(falcon.parameters())
print(f"Falcon-H1 params: {falcon.count_parameters():,}")

logits, _ = falcon(tokens)
mx.eval(logits)
print(f"Output: {logits.shape}")

## Parameter Breakdown

Let's compare where the parameters are in each architecture.

In [ ]:
for name, model in [("GPT", gpt), ("LLaMA", llama), ("Falcon-H1", falcon)]:
    total = model.count_parameters()
    embed = model.embed.weight.size
    blocks = sum(
        p.size
        for _, p in mlx.utils.tree_flatten(
            [b.parameters() for b in model.blocks]
        )
    )
    print(f"{name}: {total:,} total = {embed:,} embed + {blocks:,} blocks")

## References

- Vaswani et al. (2017). [Attention Is All You Need](https://arxiv.org/abs/1706.03762). NeurIPS 2017.
- Radford et al. (2019). [Language Models are Unsupervised Multitask Learners](https://cdn.openai.com/better-language-models/language_models_are_unsupervised_multitask_learners.pdf). OpenAI Technical Report.
- Zhang & Sennrich (2019). [Root Mean Square Layer Normalization](https://arxiv.org/abs/1910.07467). NeurIPS 2019.
- Shazeer (2020). [GLU Variants Improve Transformer](https://arxiv.org/abs/2002.05202). arXiv preprint.
- Su et al. (2021). [RoFormer: Enhanced Transformer with Rotary Position Embedding](https://arxiv.org/abs/2104.09864). Neurocomputing, 2024.
- Ainslie et al. (2023). [GQA: Training Generalized Multi-Query Transformer Models from Multi-Head Checkpoints](https://arxiv.org/abs/2305.13245). EMNLP 2023.
- Touvron et al. (2023). [LLaMA: Open and Efficient Foundation Language Models](https://arxiv.org/abs/2302.13971). arXiv preprint.
- Dao & Gu (2024). [Transformers are SSMs: Generalized Models and Efficient Algorithms Through Structured State Space Duality](https://arxiv.org/abs/2405.21060). ICML 2024.
- Zuo et al. (2025). [Falcon-H1: A Family of Hybrid-Head Language Models Redefining Efficiency and Performance](https://arxiv.org/abs/2507.22448). arXiv preprint.